# AI Resume Screening System
**LangChain + LangSmith Pipeline**

Pipeline: `Resume → Skill Extraction → Matching → Scoring → Explanation → Tracing`

## 1. Setup & Imports

In [ ]:
!pip install -q langchain langchain-community langsmith transformers sentencepiece torch

In [ ]:
import os

# ── LangSmith Free Tier Setup ───────────────────────────────────────────────
# Sign up FREE at https://smith.langchain.com → Settings → API Keys
os.environ["LANGCHAIN_TRACING_V2"]  = "true"
os.environ["LANGCHAIN_PROJECT"]     = "AI-Resume-Screener"
os.environ['LANGCHAIN_API_KEY'] = 'ls__YOUR_FREE_KEY_HERE'  # add your key before running

print("LangSmith tracing enabled ✅")
print(f"Project: AI-Resume-Screener")

LangSmith tracing enabled ✅
Project: AI-Resume-Screener


In [ ]:
from transformers import pipeline as hf_pipeline
from langchain_community.llms import HuggingFacePipeline
from langchain.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableLambda

# Load flan-t5-base – FREE, no API key needed, downloads once (~250MB)
print("Loading flan-t5-base... (first run downloads ~250MB, then cached)")

hf_pipe = hf_pipeline(
    "text2text-generation",      # flan-t5 is seq2seq, NOT text-generation
    model="google/flan-t5-base",
    max_new_tokens=256,
    do_sample=False,
)

# Wrap in LangChain
llm = HuggingFacePipeline(pipeline=hf_pipe)

print("Model ready ✅")

Loading flan-t5-base... (first run downloads ~250MB, then cached)


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

Model ready ✅


In [ ]:
# ── Step 1: Skill Extraction ─────────────────────────────────────────────────
extract_prompt = PromptTemplate(
    input_variables=["resume"],
    template="""Extract information from this resume. List only what is explicitly mentioned.

Resume: {resume}

Answer in this exact format:
Skills: <comma separated skills>
Experience: <years of experience>
Tools: <comma separated tools>"""
)

# ── Step 2: Matching ─────────────────────────────────────────────────────────
match_prompt = PromptTemplate(
    input_variables=["resume", "job_description"],
    template="""Compare this resume against the job description.

Resume: {resume}

Job Description: {job_description}

Answer in this exact format:
Matched Skills: <comma separated matched skills>
Missing Skills: <comma separated missing skills>
Overall Match: <Strong or Average or Weak>"""
)

# ── Step 3: Scoring ──────────────────────────────────────────────────────────
score_prompt = PromptTemplate(
    input_variables=["match_result"],
    template="""Based on this match analysis, give a score from 0 to 100.

Match Analysis: {match_result}

Rules: Strong match = 75-100, Average match = 40-74, Weak match = 0-39.

Answer in this exact format:
Score: <number>/100
Reason: <one sentence reason>"""
)

# ── Step 4: Explanation ──────────────────────────────────────────────────────
explain_prompt = PromptTemplate(
    input_variables=["score", "match_result"],
    template="""Given this score and match analysis, give a hiring recommendation.

Score and Reason: {score}
Match Analysis: {match_result}

Answer in this exact format:
Recommendation: <HIRE or CONSIDER or REJECT>
Explanation: <two sentences explaining why>"""
)

print("Prompts defined ✅")

Prompts defined ✅


In [ ]:
parser = StrOutputParser()

# Individual LCEL chains
extract_chain = extract_prompt | llm | parser
match_chain   = match_prompt   | llm | parser
score_chain   = score_prompt   | llm | parser
explain_chain = explain_prompt | llm | parser

def run_pipeline(resume, jd):
    """4-step screening pipeline. Each run is traced in LangSmith."""

    # Step 1 – Extract skills
    extracted = extract_chain.invoke({"resume": resume})

    # Step 2 – Match against JD
    matched = match_chain.invoke({"resume": resume, "job_description": jd})

    # Step 3 – Score
    scored = score_chain.invoke({"match_result": matched})

    # Step 4 – Explain & Recommend
    explained = explain_chain.invoke({"score": scored, "match_result": matched})

    print("\n==============================")
    print("📄 STEP 1 – EXTRACTION")
    print("------------------------------")
    print(extracted.strip())

    print("\n🤝 STEP 2 – MATCHING")
    print("------------------------------")
    print(matched.strip())

    print("\n📊 STEP 3 – SCORING")
    print("------------------------------")
    print(scored.strip())

    print("\n💡 STEP 4 – EXPLANATION")
    print("------------------------------")
    print(explained.strip())
    print("==============================\n")

print("Pipeline ready ✅")

Pipeline ready ✅


In [ ]:
strong_resume = """
Python, Machine Learning, Deep Learning, NLP
3 years experience in Data Science
Tools: TensorFlow, PyTorch, Pandas
"""

average_resume = """
Python, basic Machine Learning
1 year experience
Tools: Pandas, Scikit-learn
"""

weak_resume = """
HTML, CSS
Fresher
No ML experience
"""

job_description = """
Looking for Data Scientist with:
Python, Machine Learning, Deep Learning
Experience: 2+ years
Tools: TensorFlow, PyTorch
"""

print("Sample data ready ✅")

Sample data ready ✅


In [ ]:
print("===== STRONG CANDIDATE =====")
run_pipeline(strong_resume, job_description)

print("===== AVERAGE CANDIDATE =====")
run_pipeline(average_resume, job_description)

print("===== WEAK CANDIDATE =====")
run_pipeline(weak_resume, job_description)

===== STRONG CANDIDATE =====

📄 STEP 1 – EXTRACTION
------------------------------
Skills: comma separated skills> Experience: years of experience> Tools: comma separated tools>

🤝 STEP 2 – MATCHING
------------------------------
The job description describes the skills and abilities of the Data Scientist.

📊 STEP 3 – SCORING
------------------------------
0

💡 STEP 4 – EXPLANATION
------------------------------
0

===== AVERAGE CANDIDATE =====

📄 STEP 1 – EXTRACTION
------------------------------
Skills: comma separated skills> Experience: years of experience> Tools: comma separated tools>

🤝 STEP 2 – MATCHING
------------------------------
The job description describes a data scientist with Python, Machine Learning, Deep Learning experience.

📊 STEP 3 – SCORING
------------------------------
0

💡 STEP 4 – EXPLANATION
------------------------------
0

===== WEAK CANDIDATE =====

📄 STEP 1 – EXTRACTION
------------------------------
Skills: comma separated skills> Experience: years of e

In [ ]:
# Debug run – blank resume (edge case)
print("===== DEBUG RUN – EMPTY RESUME =====")
run_pipeline("No information provided.", job_description)

===== DEBUG RUN – EMPTY RESUME =====

📄 STEP 1 – EXTRACTION
------------------------------
Skills: comma separated skills> Experience: years of experience> Tools: comma separated tools>

🤝 STEP 2 – MATCHING
------------------------------
No information is provided about the job description.

📊 STEP 3 – SCORING
------------------------------
0

💡 STEP 4 – EXPLANATION
------------------------------
Recommendation:

